# Pretraining notebook

## Section 1: Generate text before training

Load your current model

Enter a few prompts and note the output.

Try these:
 - love is
 - baby
 - tonight

Generate 20 tokens and note the output.

Reflection: Is there any quality or meaning to the output or is it largely random text?  Why? - The text is largely random because the weights have not yet been trained.

In [1]:
import torch
import tiktoken

# This makes results more reproducible and reduces confusion when comparing outputs.
torch.manual_seed(123)

from weird_ai.model import WeirdAIModel

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

# using the text generation routine from chapter 4
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)
        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

def text_to_token_ids(text,tokenizer):
    encoded = tokenizer.encode(text, allowed_special={'<|endoftext|>'})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor
    
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

tokenizer = tiktoken.get_encoding("gpt2")

cfg = {
    "vocab_size": 50257,
    "context_length": 256,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False
}

model = WeirdAIModel(cfg["vocab_size"], cfg["context_length"], cfg["emb_dim"], cfg["n_layers"])
model.eval()

text = "love is"
token_ids = generate_text_simple(model, text_to_token_ids(text, tokenizer), 20, cfg["context_length"])
print("Output:\n", token_ids_to_text(token_ids, tokenizer))

text = "baby"
token_ids = generate_text_simple(model, text_to_token_ids(text, tokenizer), 20, cfg["context_length"])
print("Output:\n", token_ids_to_text(token_ids, tokenizer))

text = "tonight"
token_ids = generate_text_simple(model, text_to_token_ids(text, tokenizer), 20, cfg["context_length"])
print("Output:\n", token_ids_to_text(token_ids, tokenizer))

cpu
Output:
 love is Peer Spot violIGHTSictions boosters326 Mell officially protr importedship absorb scor portal allocations import shuts extr Alban
Output:
 baby Harringtonorie UV Destlen SOL 1850 moistibly Blazers Loss!!!!jan allele failed pciibly honors prototyp drain
Output:
 tonight flanked nine anomal Phill coh lasers­ sent Blazers twent profadinRender bitingstyle finger Spirits Jesuit bluntly divid


## Section 2: Understanding Logits

Using the logits starter code, do the following:
 - Apply softmax
 - Identify the highest probability token

Reflection: Why is softmax necessary before interpreting logits as probabilities? - Before Softmax, logits are not yet points on a probability distribution and are instead next token scores.

In [2]:
logits = torch.tensor([
    [1.5, 2.0, 0.5]
])

probabilities = torch.softmax(logits, dim=-1)

print(probabilities)
print(probabilities.sum())

predicted_token = torch.argmax(probabilities)

print(predicted_token)

tensor([[0.3315, 0.5465, 0.1220]])
tensor(1.0000)
tensor(1)


## Section 3: Cross Entropy Loss

### Textbook 
The textbook discussion begins on page 136

### Manually calculate
 - Probabilities
 - Log probabilities
 - Average negative log probabilities

### Compare with torch
Compare your manual results to the torch cross entropy results

```python
     torch.nn.functional.cross_entropy(...)
'''


In [3]:
import torch

# Manually calculate values

probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

# softmax ends up getting applied in torch's cross entropy function
probs = torch.softmax(probs, dim=-1)

target_index = 0

In [4]:
target_probability = probs[target_index]

print(target_probability)

tensor(0.4640)


In [5]:
log_probability = torch.log(target_probability)

print(log_probability)

tensor(-0.7679)


In [6]:
loss = -log_probability

print(loss)

tensor(0.7679)


In [7]:
# Compare with torch calculations
probs = torch.tensor([
    0.7,
    0.2,
    0.1
])

torch.nn.functional.cross_entropy(probs, torch.tensor([target_index]))

tensor(0.7679)

## Section 4: Perplexity

Compute the perplexity

```python
perplexity = torch.exp(loss)
```

**Note:** A perplexity of 10 means the model is roughly as uncertain as choosing among 10 equally likely next tokens.

In [8]:
loss = torch.tensor(2.5)

perplexity = torch.exp(loss)

print(perplexity)

tensor(12.1825)


## Section 5: Understanding the Training Loop

The training loop performs:

1. Iterate through epochs
2. Iterate through batches
3. Zero gradients
4. Calculate loss
5. Backpropagation
6. Optimizer step
7. Evaluate model
8. Generate sample text

Draw or explain this process in your own words.

# Training vs. Validation

You will:
 - Split lyrics into train/validation
 - Create loaders
 - Compute initial loss

Keep note of your: 
 - Training Loss: 
 - Validation Loss: 

In [9]:
from weird_ai.dataset import LyricsDataset
from weird_ai.config import SAMPLE_LYRICS_FILE

train_ratio = 0.9

# TODO:
# Split the corpus into train and validation text
file_path = "../data/raw/verdict.txt"
with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()
split_idx = int(train_ratio * len(text))
train_text = text[:split_idx]
val_text = text[split_idx:]

print("Training characters:", len(train_text))
print("Validation characters:", len(val_text))

Training characters: 18432
Validation characters: 2049


In [10]:
from torch.utils.data import DataLoader
# TODO:
# Create dataloaders
train_tokens = tokenizer.encode(train_text)
train_set = LyricsDataset(train_tokens, cfg["context_length"], cfg["context_length"])
train_loader = DataLoader(train_set, batch_size=2, shuffle=True, drop_last=True, num_workers=0)

val_tokens = tokenizer.encode(val_text)
val_set = LyricsDataset(val_tokens, cfg["context_length"], cfg["context_length"])
val_loader = DataLoader(val_set, batch_size=2, shuffle=False, drop_last=False, num_workers=0)

print(train_loader)
print(val_loader)

print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)
print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])

Validation loader:
torch.Size([2, 256]) torch.Size([2, 256])


In [16]:
# TODO:
# Calculate initial losses

# from chapter 5
def calc_loss_batch(input_batch, target_batch, model, device):
    input_batch = input_batch.to(device)
    target_batch = target_batch.to(device)
    logits = model(input_batch)
    loss = torch.nn.functional.cross_entropy(
        logits.flatten(0, 1), target_batch.flatten()
    )
    return loss

def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_loss_batch(
                input_batch, target_batch, model, device
            )
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

model = WeirdAIModel(cfg["vocab_size"], cfg["context_length"], cfg["emb_dim"], cfg["n_layers"])
model.eval()

initial_loss = calc_loss_loader(
        train_loader, 
        model,
        device)

with torch.no_grad():
    print("Train loss: \n", initial_loss)
    print("Validation loss: \n", calc_loss_loader(
        val_loader, 
        model,
        device))

Train loss: 
 10.97032642364502
Validation loss: 
 10.969974517822266


## Section 6: Training

Train for one Epoch
```python
train_model(...)
```

Then record the loss before and after training.  

In [21]:
num_epochs = 1

# TODO:
# Train the model
model = WeirdAIModel(cfg["vocab_size"], cfg["context_length"], cfg["emb_dim"], cfg["n_layers"])
model.train()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)
for input_batch, target_batch in train_loader:
    optimizer.zero_grad()
    loss = calc_loss_batch(input_batch, target_batch, model, device)
    loss.backward()
    optimizer.step()

model.eval()
final_loss = calc_loss_loader(train_loader, model, device)

# TODO:
# Record:
# Training loss before
# Training loss after

In [22]:
print(f"Initial loss: {initial_loss}")
print(f"Final loss: {final_loss}")

model.eval()

text = "love is"
token_ids = generate_text_simple(model, text_to_token_ids(text, tokenizer), 20, cfg["context_length"])
print("Output:\n", token_ids_to_text(token_ids, tokenizer))

Initial loss: 10.97032642364502
Final loss: 7.240394274393718
Output:
 love is,,,,,,,,,,,,,,,,,,,,


## Evaluation

Did the loss decrease? - The loss did decrease.

Why is decreasing loss important? - Decreasing loss means we're generating the right targets

## Before and After Comparison

Prompt:

love is

Before Training:

love is Peer Spot violIGHTSictions boosters326 Mell officially protr importedship absorb scor portal allocations import shuts extr Alban
__________________

After Training:
love is,,,,,,,,,,,,,,,,,,,,
__________________

Reflection:

How did the output change? - The output changed from random gibberish to just commas. Although, in one run it was a bunch of "the,"s.

What evidence do you see that the model learned something from the training data? - Outputs still don't make any sense, but the after-output is evident that the model learned that commas were common.